## Advanced Features

This notebook covers the capabilities introduced in luxy 0.1.0:

1. **Graph-style `filter()`** for traversing relationships in a single expression
2. **Response profiles** to slim down the JSON we receive
3. **Rate limiting** that scales with query complexity
4. **Query complexity** inspection
5. **Pluggable backends** (MarkLogic by default, QLever supported)
6. **Working directly with the luxql tree**

luxy is built **on top of [luxql](https://github.com/project-lux/luxql)**: every query you build is a real luxql tree, validated against the LUX advanced-search-config. There is no parallel DSL -- the luxql tree *is* the query.

### Graph-style `filter()`

Use the `__` separator to traverse relationships in a single argument. The last segment is the leaf field; the rest are relationships.

In [ ]:
from luxy import PeopleGroups, Objects

# All people who created paintings
q = PeopleGroups().filter(produced__classification__name="painting")
print(q.to_dict())
print("complexity:", q.query_complexity())

Comparator aliases (`gt`, `gte`, `lt`, `lte`, `eq`, `ne`) work the same way:

In [ ]:
Objects().filter(height__gte=1).to_dict()

And the legacy nested-dict form is unchanged:

In [ ]:
PeopleGroups().filter(produced={"classification": {"name": "painting"}}).to_dict()

### Response profiles

By default, `.get()` returns the full LUX response. Pass `profile=` to slim it down.

In [ ]:
from luxy import PROFILE_NAME, PROFILE_COUNT, PROFILE_RESULTS

count = PeopleGroups().filter(name="Rembrandt").get(profile=PROFILE_COUNT).json
print("count profile:", count)

In [ ]:
result = PeopleGroups().filter(name="Rembrandt").get(profile=PROFILE_NAME)
for item in result.json["orderedItems"][:3]:
    print(item)

### Rate limiting

luxy distinguishes between three kinds of requests:

| Bucket | Default | Used by |
|---|---|---|
| `document` | unlimited | `get_item_data` |
| `simple` | 1/second | non-relationship searches |
| `relationship` | 1/3 seconds | relationship searches |

The bucket is chosen automatically by computing the query complexity from the luxql tree.

In [ ]:
from luxy import set_rate_limits

# Aggressive (don't do this against the public API)
set_rate_limits(simple=0.0, relationship=0.0)

# Restore defaults
set_rate_limits(simple=1.0, relationship=3.0)

Plug in a load endpoint (real or future) to scale the limits dynamically. The endpoint is expected to return JSON with a `load` field; a value of `2.0` doubles the interval, `0.5` halves it.

In [ ]:
from luxy import RateLimiter, make_load_callback, REQUEST_SIMPLE

limiter = RateLimiter(
    load_callback=make_load_callback(
        "https://lux.collections.yale.edu/api/load",  # hypothetical
    ),
)
limiter.set_interval(REQUEST_SIMPLE, 1.0)

PeopleGroups().with_rate_limiter(limiter).filter(name="Rembrandt").get()

### Inspecting query complexity

Useful when you're batch-processing a long list of queries and want to back off from the more expensive ones.

In [ ]:
simple = Objects().filter(name="painting")
graph = PeopleGroups().filter(produced__classification__name="painting")

print("simple:    ", simple.query_complexity(), "->", simple._category_for_query())
print("graph hop: ", graph.query_complexity(), "->", graph._category_for_query())

### Switching backends

luxy ships with two backends:

- `MarkLogicBackend` -- the public LUX API (default).
- `QLeverBackend` -- a local SPARQL endpoint (with optional lmdb-backed result cache).

The QLever backend translates the simplest luxql shapes (text/keyword leaves, AND-of-leaves) directly to SPARQL. For anything more elaborate it transparently falls back to the MarkLogic backend you provide.

In [ ]:
from luxy import MarkLogicBackend, QLeverBackend, set_default_backend

local = QLeverBackend(
    endpoint="http://localhost:7019",
    lmdb_path="./.luxy-cache",
    fallback=MarkLogicBackend(),
)
set_default_backend(local)

# Restore the default for the rest of the notebook
set_default_backend(MarkLogicBackend())

### Working with the luxql tree directly

Every luxy query is backed by a luxql tree. You can grab it with `to_luxql()` and use any luxql primitive on it.

In [ ]:
tree = PeopleGroups().filter(produced__classification__name="painting").to_luxql()
print(tree.to_json())